# Fine-Tuning Mistral-7B for Text-to-SQL with QLoRA

Train a 7B LLM to convert plain-English questions into SQL on a **single free Colab T4 GPU**.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Pipeline: load Mistral-7B (4-bit) → attach LoRA → train on `sql-create-context` → compare base vs fine-tuned → push adapter + GGUF to your HF Hub.

## 1. Install dependencies
Unsloth manages compatible `torch` / `transformers` / `peft` / `trl` / `bitsandbytes` versions.

In [ ]:
%%capture
import torch
# Unsloth: fast QLoRA.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Pin a trl that ships SFTConfig + processing_class (avoids version drift breakage).
!pip install -q --force-reinstall --no-deps trl==0.12.2
import trl
print('cuda available:', torch.cuda.is_available(), '| trl', trl.__version__)

## 2. Configuration
Edit `HF_USERNAME` if needed. Repo names are where your trained artifacts get pushed.

In [ ]:
HF_USERNAME       = "Nagendra729"
ADAPTER_REPO      = f"{HF_USERNAME}/mistral-7b-text-to-sql-lora"
GGUF_REPO         = f"{HF_USERNAME}/mistral-7b-text-to-sql-gguf"

BASE_MODEL        = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # non-gated 4-bit mirror
MAX_SEQ_LENGTH    = 2048
LORA_RANK         = 16
MAX_TRAIN_SAMPLES = 8000   # subset keeps free-T4 run ~30-45 min; raise for better quality
MAX_STEPS         = 0      # 0 = train full subset for NUM_EPOCHS; or set e.g. 60 for a quick test
NUM_EPOCHS        = 1

## 3. Load Mistral-7B in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # auto: fp16 on T4
    load_in_4bit   = True,
)
print('loaded', BASE_MODEL)

## 4. Attach LoRA adapters
Only ~0.6% of parameters become trainable; the 7B base stays frozen in 4-bit.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_RANK,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)
model.print_trainable_parameters()

## 5. Load and format the dataset
`b-mc2/sql-create-context`: each row has a natural-language `question`, a `context` (the `CREATE TABLE` schema), and the target `answer` (SQL).

We wrap each example in a consistent instruction prompt and append the EOS token so the model learns where to stop.

In [ ]:
from datasets import load_dataset

EOS = tokenizer.eos_token

PROMPT = """You are a precise text-to-SQL engine. Given a database schema and a question, output ONLY the SQL query.

### Schema:
{schema}

### Question:
{question}

### SQL:
{sql}"""

def format_row(ex):
    text = PROMPT.format(schema=ex["context"], question=ex["question"], sql=ex["answer"]) + EOS
    return {"text": text}

ds = load_dataset("b-mc2/sql-create-context", split="train")
ds = ds.shuffle(seed=3407)

eval_raw = ds.select(range(200))                       # held-out for base-vs-tuned comparison
train_raw = ds.select(range(200, 200 + MAX_TRAIN_SAMPLES))
train_ds = train_raw.map(format_row, remove_columns=train_raw.column_names)

print('train examples:', len(train_ds))
print('\n--- sample prompt ---\n')
print(train_ds[0]['text'])

## 6. Train
`SFTTrainer` (TRL) handles supervised fine-tuning. Roughly 30–45 min on a free T4 for 8k samples.

In [ ]:
# Robust training path: pre-tokenize + plain Trainer (avoids trl SFT/padding-free churn).
# Unsloth's speedups come from the model kernels, not the trainer class.
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported

# Tokenize text -> input_ids so no raw strings reach the collator.
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,
    num_train_epochs            = NUM_EPOCHS,
    max_steps                   = MAX_STEPS if MAX_STEPS > 0 else -1,
    learning_rate               = 2e-4,
    fp16        = not is_bfloat16_supported(),
    bf16        = is_bfloat16_supported(),
    logging_steps = 10,
    optim         = "adamw_8bit",
    weight_decay  = 0.01,
    lr_scheduler_type = "linear",
    seed          = 3407,
    output_dir    = "outputs",
    report_to     = "none",
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized,
    data_collator    = collator,
    processing_class = tokenizer,
)

stats = trainer.train()
print(stats)

## 7. Evaluate: base vs. fine-tuned
We generate SQL for held-out questions and compare exact-match against the gold answer.
(The current `model` already has the trained adapter active.)

In [ ]:
import re
FastLanguageModel.for_inference(model)

INFER_PROMPT = """You are a precise text-to-SQL engine. Given a database schema and a question, output ONLY the SQL query.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""

def gen_sql(schema, question, max_new_tokens=128):
    prompt = INFER_PROMPT.format(schema=schema, question=question)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip().split("\n")[0].strip()

def norm(s):
    return re.sub(r"\s+", " ", s.strip().rstrip(";").lower())

N = 30
correct = 0
print(f"Evaluating fine-tuned model on {N} held-out questions...\n")
for i in range(N):
    ex = eval_raw[i]
    pred = gen_sql(ex["context"], ex["question"])
    ok = norm(pred) == norm(ex["answer"])
    correct += ok
    if i < 5:
        print(f"Q: {ex['question']}")
        print(f"  gold: {ex['answer']}")
        print(f"  pred: {pred}   {'✅' if ok else '❌'}\n")
print(f"Fine-tuned exact-match: {correct}/{N} = {100*correct/N:.1f}%")

## 8. Push artifacts to Hugging Face Hub
Logs you in (paste a **write** token from https://huggingface.co/settings/tokens), then pushes the LoRA adapter and a GGUF build for the demo Space.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 8a. LoRA adapter (~80 MB) — small, swappable
model.push_to_hub_merged(ADAPTER_REPO, tokenizer, save_method="lora")
print('pushed adapter ->', ADAPTER_REPO)

In [ ]:
# 8b. GGUF Q4_K_M (merged + quantized) — what the CPU demo Space loads
# Builds llama.cpp under the hood; takes a few minutes.
model.push_to_hub_gguf(GGUF_REPO, tokenizer, quantization_method="q4_k_m")
print('pushed GGUF ->', GGUF_REPO)

## Done
Your model is on the Hub:
- Adapter: `Nagendra729/mistral-7b-text-to-sql-lora`
- GGUF: `Nagendra729/mistral-7b-text-to-sql-gguf`

Next: deploy the Gradio demo in `space/` (it loads the GGUF). The GGUF filename in your repo (e.g. `mistral-7b-text-to-sql-gguf.Q4_K_M.gguf`) goes into the Space's `app.py`.